# Saving functions and testing

I coded the savings of data without the use of functions so I transfer and test into functions now

In [57]:
import pandas as pd
import numpy as np
import tempfile
from pathlib import Path
from datetime import date, datetime
from vportfolio.params import BACKTEST, CAP, LOOKBACK, TRIALS_LOG
from vportfolio.allocators import equal_weights

## Parquet files

In [7]:
# save return data to parquet file 
def save_returns(name, nav, folder = BACKTEST):
    folder.mkdir(parents=True, exist_ok=True) # creates the folder if it doesn't exist
    strat_returns = nav.pct_change(fill_method=None).dropna().to_frame(name)
    path = folder / f"{name}_returns.parquet"
    strat_returns.to_parquet(path) # in this case we need to overwrite the file if it already exists
    if not pd.read_parquet(path).equals(strat_returns): # check if the file was saved correctly
        raise ValueError(f"{name} did not save correctly")
    return path # returns the file's location for reference

In [25]:
# function testing
tmp = Path(tempfile.mkdtemp())
folder = tmp / "test_folder"
dates = pd.date_range('2020-01-01', periods=3)
nav = pd.Series([1.0, 1.1, 1.21], index=dates) # 10% return each day
path = save_returns("test", nav, folder)
saved = pd.read_parquet(path)
    
assert folder.exists() # check if the folder was created
assert path.exists() # check if the file was created
assert list(saved.columns) == ["test"] # check if the column name is correct
assert np.isclose(saved["test"], 0.1).all() # values saved

nav = pd.Series([1.0, 2.0, 3.0], index=dates)
path = save_returns("test", nav, folder)
saved = pd.read_parquet(path)
assert np.isclose(saved["test"], [1.0, 0.5]).all() # check for the exact expected returns

print("the function passed all testing")

the function passed all testing


## Data logs

In [27]:
# skeleton
#row_rw = {"date": "2026-10-28", "name": "risk_weighted", "allocator": "risk_weighted",
#            "settings": "cap=0.2, lookback=504", "role": "diagnostic",
#            "ann_return": performance_rw["ann_return"],
#            "sharpe": performance_rw["sharpe"], 
#            "turnover" : turnover_ew.mean(), 
#            "max_drawdown" : performance_ew["max_drawdown"],
#            "returns_file": "results/backtests/risk_weighted.parquet"}

In [50]:
# function to create a new row to put into the log
def make_new_row(name, allocator, role, performance, turnover, returns_file, cap = CAP, lookback = LOOKBACK):
    now = pd.Timestamp.today().strftime("%Y-%m-%d")
    row = {"date": now,
           "name": name,
           "allocator": allocator.__name__,
           "settings": f"cap={cap}, lookback={lookback}",
           "role": role,
           "ann_return": performance["ann_return"],
           "ann_vol": performance["ann_vol"],
           "sharpe": performance["sharpe"],
           "avg_turnover" : turnover.mean(), 
           "max_drawdown" : performance["max_drawdown"],
           "returns_file": str(returns_file)}                                          
    return row

In [55]:
# function testing

name = "test"
allocator = equal_weights
role = "testing"
performance = pd.Series({"ann_return": 0.01, "ann_vol": 0.1, "sharpe": 0.5, "max_drawdown": -0.2})
turnover = pd.Series([0.1,0.3])
returns_file = "results/backtests/test.parquet" # the test will only take the name so no need for it to be real

row = make_new_row(name, allocator, role, performance, turnover, returns_file)

key_list = ["date", "name", "allocator", "settings", "role", "ann_return", "ann_vol", "sharpe", "avg_turnover", "max_drawdown", "returns_file"]
# we make the list as is of today in case the function will change in the future and we forget
assert set(row.keys()) == set(key_list) # compare the names of the keys of the dictionary. use set() so order does not matter

assert row["allocator"] == "equal_weights" # that the allocator function is saved as a string
assert row["settings"] == "cap=0.2, lookback=504" # check that the right parameters are ingested
assert row["ann_return"] == 0.01
assert row["ann_vol"] == 0.1
assert row["sharpe"] == 0.5
assert row["max_drawdown"] == -0.2
assert np.isclose(row["avg_turnover"], 0.2)
assert isinstance(row["returns_file"], str)
assert row["date"] == pd.Timestamp.today().strftime("%Y-%m-%d")
print("the function passed all testing")

the function passed all testing


In [70]:
# Add new strategy runs to the trials log, replacing older rows with the same name

def update_trials_log(new_rows, path=TRIALS_LOG):
    # load the existing log, or start empty if the file is missing or empty
    if path.exists() and path.stat().st_size > 0:
        log = pd.read_csv(path)
    else:
        log = pd.DataFrame()

    # remove old rows for the strategies being written now
    if not log.empty:
        log = log[~log["name"].isin(new_rows["name"])] # new log is old log without the rows that appear in new_row

    # add the new rows below the remaining ones
    log = pd.concat([log, new_rows], ignore_index=True)

    # save the whole log, overwriting the file
    log.to_csv(path, index=False)
    return log

In [90]:
# function testing

# make a new temp every time we run the cell
tmp = Path(tempfile.mkdtemp())

# set the path and filename
log_path  = tmp / "log.csv"

# test a file is created with 1 row
test_rows1=pd.DataFrame([{"name": "A", "sharpe": 0.5}])
update_trials_log(test_rows1, path = log_path)
log = pd.read_csv(log_path)
assert len(log) == 1
assert set(log["name"]) == {"A"}

# test a new row is added
test_rows2=pd.DataFrame([{"name": "A", "sharpe": 0.5},
                        {"name":"B", "sharpe":0.2}])
update_trials_log(test_rows2, path = log_path)
log = pd.read_csv(log_path)
assert len(log) == 2
assert set(log["name"]) == {"A", "B"}

# test an old row is edited
test_rows3=pd.DataFrame([{"name": "A", "sharpe": 0.9},
                        {"name":"B", "sharpe":0.2}])
update_trials_log(test_rows3, path = log_path)
log = pd.read_csv(log_path)
assert log.loc[log["name"] == "A", "sharpe"].item() == 0.9
print("the function passed all testing")

the function passed all testing
